# Week 82, Day 4: Design interview practice

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w082d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
limit, window = 5, 60
requests = [55, 56, 57, 58, 59, 60, 61, 62, 63, 64]     # seconds
counts = {}
allowed = []
for t in requests:
    w = t // window
    if counts.get(w, 0) < limit:
        counts[w] = counts.get(w, 0) + 1
        allowed.append(t)
print(len(allowed), "allowed in", requests[-1] - requests[0] + 1, "seconds")

In [ ]:
def sliding_counter_allows(now, window, limit, counts):
    current = now // window
    elapsed = (now % window) / window                 # how far into this window we are
    estimate = counts.get(current, 0) + counts.get(current - 1, 0) * (1 - elapsed)
    if estimate < limit:
        counts[current] = counts.get(current, 0) + 1
        return True
    return False

counts = {}
allowed = [t for t in [55, 56, 57, 58, 59, 60, 61, 62, 63, 64] if sliding_counter_allows(t, 60, 5, counts)]
print(allowed)

> **Key idea.** Design discussions: clarify requirements, estimate, sketch the API and data, draw a complete simple design, dive deep into the hardest parts, then wrap up with bottlenecks and monitoring; state trade-offs aloud. Rate limiting: fixed windows are tiny but allow twice the limit at boundaries; sliding logs are exact but store every request; sliding window counters estimate current + previous × (1 - elapsed fraction) with two integers; token buckets allow bounded bursts. Across a fleet, keep counters in a fast shared store and update them atomically (increment, then compare the returned value); read-then-write races let extra requests through; batch leases cut store calls. Identify clients by API key or trusted IP (the proxy's X-Forwarded-For entry, not one the client sent); match rules by route and plan; send RateLimit headers and 429 with Retry-After; decide between failing open and closed when the store is down.

### Exercise: Fixed windows

Write a class `FixedWindow(limit, window)` with `allow(client, now)`: count requests per client per window number (`now // window`), returning `True` while the count is below the limit (and counting it) and `False` after. Old windows' counts can be thrown away. Then `max_in_any(times, span)`: given the sorted times of allowed requests, the largest number that fall within any `span`-second stretch (a request at `t` and one at `t + span` are **not** in the same stretch).

In [ ]:
class FixedWindow:
    ___

def max_in_any(times, span):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_fixed():\n    \"simple, but bursty at boundaries\"\n    f = FixedWindow(5, 60)\n    allowed = [t for t in range(55, 65) if f.allow(\"ada\", t)]\n    assert allowed == [55, 56, 57, 58, 59, 60, 61, 62, 63, 64]\n    assert f.allow(\"ada\", 65) is False and f.allow(\"bob\", 65) is True\n    assert max_in_any(allowed, 60) == 10 and max_in_any([0, 60, 120], 60) == 1 and max_in_any([], 60) == 0\n    g = FixedWindow(2, 10)\n    assert [g.allow(\"x\", t) for t in (0, 1, 2, 9, 10, 11, 12)] == [True, True, False, False, True, True, False]")

### Exercise: A sliding window log

Write a class `SlidingLog(limit, window)` with `allow(client, now)`: keep each client's timestamps of **allowed** requests (in a `deque`), drop those at or before `now - window`, and allow (recording the time) if fewer than `limit` remain. `stored()` returns the total number of timestamps held, to show the memory cost.

In [ ]:
from collections import deque

class SlidingLog:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_log():\n    \"exact, at a memory cost\"\n    s = SlidingLog(5, 60)\n    allowed = [t for t in range(55, 65) if s.allow(\"ada\", t)]\n    assert allowed == [55, 56, 57, 58, 59]\n    assert s.allow(\"ada\", 115) is True and s.allow(\"ada\", 115.5) is False and s.allow(\"ada\", 116) is True\n    for i in range(100):\n        s.allow(f\"user{i}\", 200)\n    assert s.stored() == 105")

### Exercise: A sliding window counter

Write a class `SlidingCounter(limit, window)` with `allow(client, now)`: the client's current window is `now // window`, and the fraction of it that has passed is `(now % window) / window`. Estimate the requests in the last `window` seconds as `current_count + previous_count × (1 - fraction)`; allow (and count) if the estimate is below the limit. Keep only the current and previous window per client. Then `remaining(client, now)`: the whole number of requests still allowed now (`limit - estimate`, rounded down, at least 0), without counting anything.

In [ ]:
import math

class SlidingCounter:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_sliding():\n    \"almost exact, two numbers each\"\n    s = SlidingCounter(5, 60)\n    allowed = [t for t in range(55, 65) if s.allow(\"ada\", t)]\n    assert allowed == [55, 56, 57, 58, 59, 61], \"one second into the next window, the estimate dips just below the limit\"\n    assert s.remaining(\"ada\", 90) == 1 and s.allow(\"ada\", 90) and s.allow(\"ada\", 91)\n    assert s.allow(\"ada\", 92) is False and s.remaining(\"ada\", 92) == 0\n    assert s.remaining(\"ada\", 150) == 3 and s.remaining(\"ada\", 300) == 5 and s.remaining(\"new\", 0) == 5")

### Exercise: A shared store, done atomically

A shared store is simulated by `Store`: `get(key)`, `set(key, value)` and `incr(key)` (atomic: adds 1 and returns the new value). Write `racy_allow(store, key, limit)` (read, compare, then write: the wrong way) and `atomic_allow(store, key, limit)` (increment first, then compare the returned value: allow if it's at most the limit). Then `interleave(store, key, limit, servers)`: simulate `servers` API servers each handling one request **at the same moment** with the racy method: all read first, then all decide and write. Return how many were allowed. And `leased(store, key, limit, batch)`: a class-free helper returning how many of the remaining requests one server may serve locally by leasing up to `batch` at once: atomically add `batch` with repeated `incr` calls... more simply, use `store.incr_by(key, n)` (atomic, returns the new value) and return `max(0, min(batch, limit - (new - batch)))`.

In [ ]:
class Store:
    def __init__(self):
        self.data = {}
    def get(self, key):
        return self.data.get(key, 0)
    def set(self, key, value):
        self.data[key] = value
    def incr(self, key):
        return self.incr_by(key, 1)
    def incr_by(self, key, n):
        self.data[key] = self.data.get(key, 0) + n
        return self.data[key]

def racy_allow(store, key, limit):
    ___

def atomic_allow(store, key, limit):
    ___

def interleave(store, key, limit, servers):
    ___

def leased(store, key, limit, batch):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_atomic():\n    \"increment, then compare\"\n    s = Store()\n    assert [atomic_allow(s, \"k\", 3) for _ in range(5)] == [True, True, True, False, False]\n    r = Store()\n    assert [racy_allow(r, \"k\", 3) for _ in range(5)] == [True, True, True, False, False], \"fine one at a time\"\n\ndef test_race():\n    \"the race lets extra requests through\"\n    s = Store()\n    s.set(\"k\", 4)\n    assert interleave(s, \"k\", 5, servers=3) == 3 and s.get(\"k\") == 5\n    a = Store()\n    a.set(\"k\", 4)\n    assert sum(atomic_allow(a, \"k\", 5) for _ in range(3)) == 1\n\ndef test_leases():\n    \"batches cut store calls\"\n    s = Store()\n    assert leased(s, \"k\", 25, 10) == 10 and leased(s, \"k\", 25, 10) == 10 and leased(s, \"k\", 25, 10) == 5\n    assert leased(s, \"k\", 25, 10) == 0")

### Exercise: Rules, identity and headers

Write `client_id(headers, remote_addr, trusted_proxies)`: the API key from `Authorization: Bearer KEY` as `"key:KEY"` if present; otherwise, if the connection came from a trusted proxy (its address is in `trusted_proxies`), the **last** address in `X-Forwarded-For` (comma-separated, stripped) as `"ip:ADDRESS"`; otherwise `"ip:" + remote_addr`. Then `pick_rule(rules, path, plan)`: `rules` is a list of `(pattern, plan, limit, window)` where `pattern` is a path prefix and `plan` is a plan name or `"*"`; return the matching rule with the **longest** prefix, preferring an exact plan match over `"*"` at the same length; `None` if nothing matches. And `headers(limit, remaining, reset)`: the dict `{"RateLimit-Limit": ..., "RateLimit-Remaining": ..., "RateLimit-Reset": ...}` with string values, plus `"Retry-After"` (the reset) when `remaining` is 0.

In [ ]:
def client_id(headers, remote_addr, trusted_proxies):
    ___

def pick_rule(rules, path, plan):
    ___

def headers(limit, remaining, reset):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_identity():\n    \"keys, trusted proxies, and forged headers\"\n    proxies = {\"10.0.0.5\"}\n    assert client_id({\"Authorization\": \"Bearer abc123\"}, \"10.0.0.5\", proxies) == \"key:abc123\"\n    assert client_id({\"X-Forwarded-For\": \"6.6.6.6, 203.0.113.9\"}, \"10.0.0.5\", proxies) == \"ip:203.0.113.9\"\n    assert client_id({\"X-Forwarded-For\": \"1.2.3.4\"}, \"198.51.100.7\", proxies) == \"ip:198.51.100.7\"\n    assert client_id({}, \"198.51.100.7\", proxies) == \"ip:198.51.100.7\"\n\ndef test_rules():\n    \"the most specific rule\"\n    rules = [(\"/\", \"*\", 100, 60), (\"/search\", \"*\", 10, 60), (\"/search\", \"partner\", 100, 60), (\"/tickets\", \"free\", 50, 60)]\n    assert pick_rule(rules, \"/search?q=x\", \"free\") == (\"/search\", \"*\", 10, 60)\n    assert pick_rule(rules, \"/search\", \"partner\") == (\"/search\", \"partner\", 100, 60)\n    assert pick_rule(rules, \"/tickets/7\", \"free\") == (\"/tickets\", \"free\", 50, 60) and pick_rule(rules, \"/tickets/7\", \"partner\") == (\"/\", \"*\", 100, 60)\n    assert pick_rule([(\"/api\", \"*\", 1, 1)], \"/other\", \"free\") is None\n\ndef test_headers():\n    \"tell clients their limits\"\n    assert headers(100, 37, 21) == {\"RateLimit-Limit\": \"100\", \"RateLimit-Remaining\": \"37\", \"RateLimit-Reset\": \"21\"}\n    assert headers(10, 0, 42)[\"Retry-After\"] == \"42\"")

### Exercise: Comparing on real traffic

Write `compare(limiters, trace)`: `limiters` maps a name to a fresh limiter object with `allow(client, now)`, and `trace` is a list of `(client, time)` requests. Run every limiter over the trace and return a dict from name to `(allowed, worst_minute)`, where `worst_minute` is the most requests any one client had allowed within any 60-second stretch (use the `max_in_any` idea from the first exercise; it's given here). This shows the boundary burst in numbers.

In [ ]:
import bisect

def max_in_any(times, span):
    best = 0
    for i, t in enumerate(times):
        best = max(best, bisect.bisect_left(times, t + span) - i)
    return best

def compare(limiters, trace):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_compare():\n    \"the boundary burst, measured\"\n    class Fixed:\n        def __init__(self):\n            self.c = {}\n        def allow(self, client, now):\n            k = (client, now // 60)\n            if self.c.get(k, 0) < 5:\n                self.c[k] = self.c.get(k, 0) + 1\n                return True\n            return False\n    class Log:\n        def __init__(self):\n            self.l = {}\n        def allow(self, client, now):\n            log = [t for t in self.l.get(client, []) if t > now - 60]\n            ok = len(log) < 5\n            if ok:\n                log.append(now)\n            self.l[client] = log\n            return ok\n    trace = [(\"ada\", t) for t in range(50, 70)] + [(\"bob\", t) for t in range(0, 200, 10)]\n    assert compare({\"fixed\": Fixed(), \"log\": Log()}, trace) == {\"fixed\": (27, 10), \"log\": (22, 5)}")

### Bug hunt: Double the limit at midnight

A "3 per minute" limit uses fixed windows, so a client sending at the end of one minute and the start of the next gets six through in two seconds: `allowed: 6`. Use a sliding window log of the last 60 seconds instead, so it prints `allowed: 3`.

In [ ]:
limit, window = 3, 60
counts = {}

def allow(now):
    w = now // window
    if counts.get(w, 0) < limit:
        counts[w] = counts.get(w, 0) + 1
        return True
    return False

times = [59.0, 59.3, 59.6, 60.0, 60.3, 60.6]
print("allowed:", sum(allow(t) for t in times))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a sliding window\"\n    expect_output(\"allowed: 3\")")

### Bug hunt: A forged address

The limiter trusts the **first** `X-Forwarded-For` entry, which the client wrote itself, so an attacker gets a fresh limit by sending a new fake address each time: `distinct clients: 3`. Our proxy appends the real address as the **last** entry; use that, so it prints `distinct clients: 1`.

In [ ]:
requests = ["1.1.1.1, 203.0.113.9", "2.2.2.2, 203.0.113.9", "3.3.3.3, 203.0.113.9"]

def client(forwarded):
    return forwarded.split(",")[0].strip()

print("distinct clients:", len({client(r) for r in requests}))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"trust only your own proxy's entry\"\n    expect_output(\"distinct clients: 1\")")

### Bug hunt: A race between servers

Two servers check the same counter at the same moment by reading it, comparing, and writing it back, so both see 4, both allow, and the limit of 5 lets 6 through: `allowed: 6`. Use the store's atomic increment and compare the value it returns, so it prints `allowed: 5`.

In [ ]:
store = {"k": 4}
allowed = 4

def check():
    count = store["k"]
    return count

seen_a, seen_b = check(), check()          # both servers read before either writes
for seen in (seen_a, seen_b):
    if seen < 5:
        store["k"] = seen + 1
        allowed += 1
print("allowed:", allowed)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"atomic increment\"\n    expect_output(\"allowed: 5\")")

### Puzzle: Checking a request

Arrange a rate-limit check that identifies the client, finds the rule, updates a shared counter atomically, and answers with headers. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the most requests a fixed "5 per minute" window can let through in 2 seconds
q = ___   # the sliding counter estimate 30 s into a window with 2 now and 4 in the previous window
r = ___   # the status code for a refused request
s = ___   # "open" or "closed": failing ___ keeps the API up when the limiter's store is down
t = ___   # which X-Forwarded-For entry to trust when your proxy appends: "first" or "last"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 10 and q == 4 and r == 429, \"p, q and r\"\n    assert s == \"open\" and t == \"last\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a rate-limiting service

Write a class `RateLimiter(rules, store, trusted_proxies, exempt=(), fail_open=True)` using the given `client_id`, `pick_rule` and `headers`. `rules` are `(prefix, plan, limit, window)`; `store` has `incr_window(key, window_number)` (atomic: adds 1 to that key's count for that window and returns it) and `get_window(key, window_number)`, and either may raise `ConnectionError` when the store is down. `check(request, plan, now)` takes a request dict with `path`, `headers` and `remote_addr`, and returns `(status, response_headers)`: exempt clients (their `client_id` is in `exempt`) get `(200, {})` without touching the store; no matching rule also gives `(200, {})`. Otherwise, with the sliding window counter method on key `CLIENT|PREFIX`: read the previous window's count (`get_window`), increment the current one (`incr_window`), estimate `current + previous × (1 - fraction)` (counting this request), and if the estimate is **more than** the limit refuse with `429`, else `200`; headers come from `headers(limit, max(0, floor(limit - estimate)), reset)`, where `reset` is the whole seconds until the current window ends (`window - now % window`, rounded up). If the store raises `ConnectionError`, return `(200, {"RateLimit-Policy": "unenforced"})` when `fail_open`, else `(503, {})`. Count each outcome in `self.counts` (keys `"allowed"`, `"limited"`, `"exempt"`, `"unenforced"`, `"unavailable"`).

In [ ]:
import math

def client_id(headers, remote_addr, trusted_proxies):
    auth = headers.get("Authorization", "")
    if auth.startswith("Bearer ") and auth[7:]:
        return "key:" + auth[7:]
    forwarded = headers.get("X-Forwarded-For")
    if remote_addr in trusted_proxies and forwarded:
        return "ip:" + forwarded.split(",")[-1].strip()
    return "ip:" + remote_addr

def pick_rule(rules, path, plan):
    matching = [r for r in rules if path.startswith(r[0]) and r[1] in (plan, "*")]
    return max(matching, key=lambda r: (len(r[0]), r[1] == plan)) if matching else None

def headers(limit, remaining, reset):
    out = {"RateLimit-Limit": str(limit), "RateLimit-Remaining": str(remaining), "RateLimit-Reset": str(reset)}
    if remaining == 0:
        out["Retry-After"] = str(reset)
    return out

class RateLimiter:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("class Store:\n    def __init__(self):\n        self.data, self.down, self.calls = {}, False, 0\n    def incr_window(self, key, window_number):\n        self.calls += 1\n        if self.down:\n            raise ConnectionError(\"store down\")\n        self.data[(key, window_number)] = self.data.get((key, window_number), 0) + 1\n        return self.data[(key, window_number)]\n    def get_window(self, key, window_number):\n        self.calls += 1\n        if self.down:\n            raise ConnectionError(\"store down\")\n        return self.data.get((key, window_number), 0)\n\nRULES = [(\"/\", \"*\", 100, 60), (\"/search\", \"*\", 3, 60), (\"/search\", \"partner\", 30, 60)]\n\ndef req(path=\"/search\", key=\"abc\"):\n    return {\"path\": path, \"headers\": {\"Authorization\": f\"Bearer {key}\"} if key else {}, \"remote_addr\": \"198.51.100.7\"}\n\ndef test_limits():\n    \"sliding counters with headers\"\n    rl = RateLimiter(RULES, Store(), {\"10.0.0.5\"})\n    results = [rl.check(req(), \"free\", 10) for _ in range(4)]\n    assert [s for s, _ in results] == [200, 200, 200, 429]\n    assert results[0][1] == {\"RateLimit-Limit\": \"3\", \"RateLimit-Remaining\": \"2\", \"RateLimit-Reset\": \"50\"}\n    assert results[3][1][\"Retry-After\"] == \"50\"\n    assert rl.check(req(), \"partner\", 10)[0] == 200 and rl.check(req(key=\"other\"), \"free\", 10)[0] == 200\n    assert rl.check(req(), \"free\", 90)[0] == 429\n    status, hdrs = rl.check(req(), \"free\", 115)\n    assert status == 200 and hdrs[\"RateLimit-Remaining\"] == \"0\" and hdrs[\"RateLimit-Reset\"] == \"5\"\n    assert rl.check(req(path=\"/tickets\"), \"free\", 115)[1][\"RateLimit-Limit\"] == \"100\"\n    assert rl.counts == {\"allowed\": 7, \"limited\": 2}\n\ndef test_exempt_and_store_down():\n    \"exemptions, failing open and closed\"\n    store = Store()\n    rl = RateLimiter(RULES, store, set(), exempt={\"key:healthcheck\"})\n    assert rl.check(req(key=\"healthcheck\"), \"free\", 0) == (200, {}) and store.calls == 0\n    assert rl.check({\"path\": \"/x\", \"headers\": {}, \"remote_addr\": \"1.2.3.4\"}, \"free\", 0)[0] == 200\n    store.down = True\n    assert rl.check(req(), \"free\", 0) == (200, {\"RateLimit-Policy\": \"unenforced\"})\n    strict = RateLimiter(RULES, store, set(), fail_open=False)\n    assert strict.check(req(), \"free\", 0) == (503, {})\n    assert rl.counts == {\"exempt\": 1, \"allowed\": 1, \"unenforced\": 1} and strict.counts == {\"unavailable\": 1}")

## Warm-up

A "5 per minute" limit with fixed windows let **ten** requests through in ten seconds: five at the end of one minute and five at the start of the next. That boundary burst is the classic flaw of the simplest algorithm, and today's first lesson in why design choices need checking against real traffic.

## How design discussions go

A design question is a conversation, usually about 45 minutes, and a steady structure keeps it on track:

1. **Clarify** (5 minutes): what does it do, for whom, and at what scale? Ask; don't assume. Write the functional and non-functional requirements down (Week 81's URL shortener).
2. **Estimate** (5 minutes): requests per second, storage, read-to-write ratio. Round numbers; say your assumptions aloud.
3. **API and data** (5 minutes): the main endpoints or functions, and what's stored.
4. **High-level design** (10 minutes): the boxes and arrows: clients, load balancers, services, caches, queues, databases. Get something complete and simple on the board first.
5. **Deep dives** (15 minutes): the two or three hardest parts, in detail: the algorithm, the data model under load, a failure mode. This is where most of the judgement is.
6. **Wrap up** (5 minutes): the bottlenecks, what you'd monitor, what you'd do next with more time.

Throughout: **state trade-offs out loud** ("a sliding log is exact but stores every request; at our scale that's 2 GB, so I'd use the counter approximation"), check in with the other person, and prefer simple, well-understood pieces over clever ones.

## The problem: requirements and numbers

"Design a rate limiter for our public API."

**Clarified requirements**:

- limit each client (by API key, or by IP address for anonymous calls) to a number of requests per window, with **different limits per route and per plan** (`/search` is expensive: 10 a minute; `/tickets`: 100 a minute; partners get ten times more);
- shared across **all** API servers: the limit is per client, not per client per server;
- adds very little latency (a millisecond or two at most), and doesn't become the thing that takes the API down;
- tells clients their limits in **headers**, and how long to wait when refused (`429` with `Retry-After`).

**Numbers**: 50,000 requests a second at peak across 40 API servers, 1 million active API keys. Each check is one read and write of a small counter, so the shared store must handle about 50,000 operations a second with millisecond latency: that's an in-memory store (Redis), not the main database. Memory: a few counters per active key, tens of bytes each, so tens or hundreds of megabytes: trivial.

## Algorithms

| Algorithm | How it works | Strengths | Weaknesses |
|---|---|---|---|
| **Fixed window** | a counter per client per window (`t // 60`); refuse once it reaches the limit | tiny state, one atomic increment | bursts of up to twice the limit at window boundaries |
| **Sliding window log** | store the timestamp of every allowed request; count those in the last 60 s | exact | memory grows with the limit; expensive at high limits |
| **Sliding window counter** | keep this window's and the previous window's counts; estimate the last 60 s as `current + previous × (fraction of the previous window still inside)` | near-exact, tiny state | an approximation (assumes requests were spread evenly in the previous window) |
| **Token bucket** (Week 81) | tokens refill at a steady rate up to a capacity; each request takes one | allows controlled bursts, smooth average | two numbers to tune; refill arithmetic per check |

The sliding window counter is a popular default for API limits: two integers per client and almost no boundary burst. The token bucket suits limits where short bursts are fine but the long-run rate isn't.

## Across a fleet, and the details

**Shared state.** With 40 servers, each keeping its own counters would allow 40 times the limit. The counters live in a **shared store**, and the check must be **atomic**: "read the count, compare, write count + 1" done as three separate steps lets two servers both read 4, both decide "under 5", and both write 5, letting the sixth request through. Stores provide atomic operations (Redis's `INCR`, or a small script run atomically), which return the **new** value, so the check becomes "increment, then compare what came back".

**Latency and load.** One store round trip per request is usually fine (under a millisecond in the same data centre). For very high traffic, servers can take tokens in **batches** (lease 10 at a time and spend them locally), trading a little accuracy for far fewer store calls.

**Identity.** Limit by API key when there is one; otherwise by client IP. Behind a load balancer, the connection comes from the balancer, so the client's address is in `X-Forwarded-For`, which a client can also forge: trust only the entry your own proxy added (the **last** one, when your proxy appends).

**Rules.** A small table: route pattern, plan, limit, window; the most specific matching rule wins. Some clients (your own health checks) are exempt.

**Responses.** Every response carries `RateLimit-Limit`, `RateLimit-Remaining` and `RateLimit-Reset` (seconds until the window resets), and a refusal is `429 Too Many Requests` with `Retry-After`. Well-behaved clients slow down without guessing.

**When the store is down.** **Fail open** (allow requests unlimited for a while) keeps the API up but unprotected; **fail closed** (refuse) protects the backend but takes the API down with the limiter. For a public API, failing open with a local, approximate fallback limit is the usual choice; say which you'd choose and why.

## Practice

## Quiz

## Daily challenge

## Before you go

Practise the structure out loud on a different problem for fifteen minutes ("design a parking garage's booking system" works): clarify, estimate, API, high-level design, one deep dive, wrap-up. Tomorrow: a second practice problem, **a chat system**, with ordering, delivery receipts and people who go offline.